In [1]:
import os
import re
import joblib
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks, optimizers
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

# Set random seed for reproducibility
tf.random.set_seed(42)
np.random.seed(42)

# Define paths
REVIEWS_PATH = r"C:\Users\Prasanth Rajaram\OneDrive\Desktop\project\InsureAI\data\raw\customer_reviews.csv"
BASE_DIR = r"C:\Users\Prasanth Rajaram\InsureAI_Local"
MODEL_DIR = os.path.join(BASE_DIR, "models")
REPORTS_DIR = os.path.join(BASE_DIR, "reports")
os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(REPORTS_DIR, exist_ok=True)

# List of common English stopwords to clean text without external downloads (negations preserved)
STOPWORDS = set([
    "i", "me", "my", "myself", "we", "our", "ours", "ourselves", "you", "your", "yours", "yourselves",
    "he", "him", "his", "himself", "she", "her", "hers", "herself", "it", "its", "itself", "they",
    "them", "their", "theirs", "themselves", "what", "which", "who", "whom", "this", "that", "these",
    "those", "am", "is", "are", "was", "were", "be", "been", "being", "have", "has", "had", "having",
    "do", "does", "did", "doing", "a", "an", "the", "and", "but", "if", "or", "because", "as", "until",
    "while", "of", "at", "by", "for", "with", "about", "between", "into", "through",
    "during", "before", "after", "above", "below", "to", "from", "up", "down", "in", "out", "on",
    "off", "over", "under", "again", "further", "then", "once", "here", "there", "when", "where",
    "why", "how", "all", "any", "both", "each", "few", "more", "most", "other", "some", "such",
    "only", "own", "same", "so", "than", "too", "s", "t", "can",
    "will", "just", "should", "now"
])

def clean_text(text):
    """
    Cleans text by converting to lowercase, removing punctuation, and filtering out stopwords.
    """
    if not isinstance(text, str):
        return ""
    # Lowercase
    text = text.lower()
    # Remove punctuation & numbers
    text = re.sub(r'[^a-z\s]', '', text)
    # Remove stopwords
    words = text.split()
    cleaned_words = [w for w in words if w not in STOPWORDS]
    return " ".join(cleaned_words)

def build_lstm_model(vocab_size=10000, embedding_dim=128):
    """
    Constructs the LSTM network model for binary sentiment classification.
    Embedding layer -> SpatialDropout1D -> LSTM layer -> Dense output.
    """
    model = models.Sequential(name="LSTM_Sentiment_Classifier")
    
    # 1. Embedding Layer: maps token integers to dense vector representation (with masking enabled)
    model.add(layers.Embedding(input_dim=vocab_size, output_dim=embedding_dim, mask_zero=True, name="embedding"))
    
    # 2. Spatial Dropout: drops entire 1D feature maps to prevent overfitting in NLP sequences
    model.add(layers.SpatialDropout1D(0.2, name="spatial_dropout"))
    
    # 3. LSTM Layer: processes text sequence dependencies
    model.add(layers.LSTM(64, dropout=0.2, name="lstm"))
    
    # 4. Output Layer: Single unit with sigmoid activation to output sentiment probability [0, 1]
    model.add(layers.Dense(1, activation='sigmoid', name='output'))
    
    return model

def main():
    print("=" * 80)
    print("MODULE 4: RNN / LSTM SENTIMENT ANALYSIS PIPELINE")
    print("=" * 80)
    
    # 1. Load data
    print(f"\nLoading reviews from: {REVIEWS_PATH}")
    df = pd.read_csv(REVIEWS_PATH)
    print(f"Loaded Shape: {df.shape}")
    
    # Remove rows with null reviews or labels
    df = df.dropna(subset=['review_text', 'sentiment'])
    
    # Filter out neutral reviews for binary sentiment classification
    df_binary = df[df['sentiment'].isin(['positive', 'negative'])].copy()
    print(f"Filtered Shape (Binary - Positive/Negative): {df_binary.shape}")
    
    # Label mapping (positive -> 1, negative -> 0)
    df_binary['label'] = df_binary['sentiment'].map({'positive': 1, 'negative': 0})
    
    # 2. Clean reviews
    print("\nCleaning text reviews...")
    df_binary['cleaned_text'] = df_binary['review_text'].apply(clean_text)
    
    # 3. Train/Test Split
    X_train_text, X_test_text, y_train, y_test = train_test_split(
        df_binary['cleaned_text'].values,
        df_binary['label'].values,
        test_size=0.2,
        random_state=42,
        stratify=df_binary['label'].values
    )
    print(f"Split completed: Train={len(X_train_text)}, Test={len(X_test_text)}")
    
    # 4. Tokenization and Padding
    vocab_size = 10000
    max_length = 100
    
    print("\nTokenizing and padding sequences...")
    tokenizer = Tokenizer(num_words=vocab_size, oov_token="<OOV>")
    tokenizer.fit_on_texts(X_train_text)
    
    # Convert text to padded sequences (using 'pre' padding to preserve final state output)
    X_train_seq = tokenizer.texts_to_sequences(X_train_text)
    X_train_padded = pad_sequences(X_train_seq, maxlen=max_length, padding='pre', truncating='pre')
    
    X_test_seq = tokenizer.texts_to_sequences(X_test_text)
    X_test_padded = pad_sequences(X_test_seq, maxlen=max_length, padding='pre', truncating='pre')
    
    # Save the tokenizer to disk for future predictions
    tokenizer_path = os.path.join(MODEL_DIR, "sentiment_tokenizer.joblib")
    joblib.dump(tokenizer, tokenizer_path)
    print(f"Fitted Tokenizer saved to: {tokenizer_path}")
    
    # 5. Build Model
    print("\nBuilding LSTM Model...")
    model = build_lstm_model(vocab_size=vocab_size, embedding_dim=128)
    
    model.compile(
        optimizer=optimizers.Adam(learning_rate=0.001),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    
    model.summary()
    
    # 6. Train Model
    print("\nTraining LSTM Model...")
    early_stopping = callbacks.EarlyStopping(
        monitor='val_loss',
        patience=3,
        restore_best_weights=True,
        verbose=1
    )
    
    history = model.fit(
        X_train_padded, y_train,
        validation_split=0.2,
        epochs=10,
        batch_size=64,
        callbacks=[early_stopping],
        verbose=1
    )
    
    # 7. Evaluate Model
    print("\n" + "=" * 50)
    print("MODEL EVALUATION ON TEST SET")
    print("=" * 50)
    
    y_pred_probs = model.predict(X_test_padded).flatten()
    y_pred = (y_pred_probs >= 0.5).astype(int)
    
    test_acc = accuracy_score(y_test, y_pred)
    test_f1 = f1_score(y_test, y_pred)
    
    print(f"Test Accuracy: {test_acc:.4f}")
    print(f"Test F1-Score: {test_f1:.4f}\n")
    
    print("Classification Report:")
    print(classification_report(y_test, y_pred, target_names=['Negative', 'Positive'], zero_division=0))
    
    print("Confusion Matrix:")
    print(confusion_matrix(y_test, y_pred))
    
    # 8. Test custom review sentences
    print("\n" + "=" * 50)
    print("VERIFYING WITH EXAMPLE SENTENCES")
    print("=" * 50)
    
    test_sentences = [
        "My claim was settled very fast",
        "I had a terrible experience with customer service",
        "The response was decent, but they took a long time to pay out.",
        "Absolutely amazing support and friendly agents!",
        "They rejected my claim without any solid explanation."
    ]
    
    cleaned_test = [clean_text(s) for s in test_sentences]
    test_seq = tokenizer.texts_to_sequences(cleaned_test)
    test_padded = pad_sequences(test_seq, maxlen=max_length, padding='pre', truncating='pre')
    
    preds_probs = model.predict(test_padded).flatten()
    
    for s, prob in zip(test_sentences, preds_probs):
        pred_label = "Positive" if prob >= 0.5 else "Negative"
        print(f"Review: '{s}'")
        print(f"  Predicted Sentiment: {pred_label} (Confidence: {prob:.4f})\n")
        
    # 9. Save Best Model
    model_save_path = os.path.join(MODEL_DIR, "sentiment_lstm_model.keras")
    model.save(model_save_path)
    print(f"Saved trained LSTM model to: {model_save_path}\n")

if __name__ == "__main__":
    main()


MODULE 4: RNN / LSTM SENTIMENT ANALYSIS PIPELINE

Loading reviews from: C:\Users\Prasanth Rajaram\OneDrive\Desktop\project\InsureAI\data\raw\customer_reviews.csv
Loaded Shape: (10147, 3)
Filtered Shape (Binary - Positive/Negative): (7061, 3)

Cleaning text reviews...
Split completed: Train=5648, Test=1413

Tokenizing and padding sequences...
Fitted Tokenizer saved to: C:\Users\Prasanth Rajaram\InsureAI_Local\models\sentiment_tokenizer.joblib

Building LSTM Model...


Model: "LSTM_Sentiment_Classifier"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ spatial_dropout                 │ ?                      │             0 │
│ (SpatialDropout1D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ output (Dense)                  │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)


Training LSTM Model...
Epoch 1/10
71/71 ━━━━━━━━━━━━━━━━━━━━ 29s 221ms/step - accuracy: 0.8342 - loss: 0.4087 - val_accuracy: 0.9460 - val_loss: 0.1576
Epoch 2/10
71/71 ━━━━━━━━━━━━━━━━━━━━ 13s 180ms/step - accuracy: 0.9790 - loss: 0.0835 - val_accuracy: 0.9593 - val_loss: 0.1095
Epoch 3/10
71/71 ━━━━━━━━━━━━━━━━━━━━ 13s 184ms/step - accuracy: 0.9958 - loss: 0.0232 - val_accuracy: 0.9628 - val_loss: 0.1234
Epoch 4/10
71/71 ━━━━━━━━━━━━━━━━━━━━ 13s 177ms/step - accuracy: 0.9987 - loss: 0.0086 - val_accuracy: 0.9513 - val_loss: 0.1843
Epoch 5/10
71/71 ━━━━━━━━━━━━━━━━━━━━ 14s 191ms/step - accuracy: 0.9989 - loss: 0.0073 - val_accuracy: 0.9584 - val_loss: 0.1534
Epoch 5: early stopping
Restoring model weights from the end of the best epoch: 2.

MODEL EVALUATION ON TEST SET
45/45 ━━━━━━━━━━━━━━━━━━━━ 3s 52ms/step
Test Accuracy: 0.9611
Test F1-Score: 0.9636

Classification Report:
              precision    recall  f1-score   support

    Negative       0.97      0.95      0.96       663
 